In [26]:
import json
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

class CustomDense(layers.Layer):

    def __init__(self, units, activation=None, dropout_rate=0.0, **kwargs):
        super(CustomDense, self).__init__(**kwargs)

        self.units = units
        self.activation = tf.keras.activations.get(activation)
        self.dropout_rate = dropout_rate
        self.dropout = layers.Dropout(dropout_rate)

    def build(self, input_shape):

        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='he_normal',
            trainable=True,
            name='kernel'
        )

        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True,
            name='bias'
        )

    def call(self, inputs, training=False):

        x = tf.matmul(inputs, self.w) + self.b

        if self.activation is not None:
            x = self.activation(x)

        x = self.dropout(x, training=training)

        return x

    def get_config(self):

        config = super().get_config()

        config.update({
            "units": self.units,
            "activation": tf.keras.activations.serialize(self.activation),
            "dropout_rate": self.dropout_rate
        })

        return config


try:
    model = tf.keras.models.load_model('model_strukly.keras', custom_objects={'CustomDense': CustomDense})
    with open('penerjemah_kategori.pkl', 'rb') as file:
        encoder = pickle.load(file)

    print("Model berhasil dimuat!")

except Exception as e:
    print(f"Error load model: {e}")


def predict_category(text, threshold=0.75):
    input_tensor = tf.constant([text])
    prediction = model(input_tensor, training=False).numpy()[0]
    class_id = int(np.argmax(prediction))
    confidence = float(prediction[class_id])
    category = encoder.inverse_transform([class_id])[0]
    need_review = confidence < threshold
    output = {
        "class_id": class_id,
        "category": category,
        "confidence": round(confidence, 2),
        "need_review": bool(need_review)}

    return output

Model berhasil dimuat!


In [24]:
if __name__ == "__main__":
    teks_noisy = [  "Kentang goreng",
                    "TELUR OMEGA3",
                    "TLR OMEGA",
                    "365 FACIAL TISSUE",
                    "PULPEN STANDARD",
                    "KERANJANG SAMPAH 50L",
                    "WALLS PADLE POP 55ML LEMON TEA/36",
                    "INDOMI SOTO MIE",
                    "INDOMIESOTO MIE",
                    "IDM STO MIE",
                    "ULTRA MILK 200CARAMEL/PCS",
                    "BIMOLI MINYAK GORENG",
                    "BML MNYK GRG",
                    "KIRANTI SEHAT BLN",
                    "SOKLIN PEMUTIH MTH LEMON 1 L",
                    "BRS PULEN WANGI",
                    "REMOTE AC",
                    "BAJU KOKO XL",
                    "MUKENA POLKADOT",
                    "HANSAPLAST",
                    "MINYAK TELON",
                    "MEJA BELAJAR",
                    "LAMPU KUNING 18W"
                    ]

    for i in teks_noisy:
        hasil_json = predict_category(i)
        print(i,hasil_json,"\n")

Kentang goreng {'class_id': 3, 'category': 'Makanan & Bahan Makanan', 'confidence': 0.98, 'need_review': False} 

TELUR OMEGA3 {'class_id': 3, 'category': 'Makanan & Bahan Makanan', 'confidence': 0.96, 'need_review': False} 

TLR OMEGA {'class_id': 3, 'category': 'Makanan & Bahan Makanan', 'confidence': 0.89, 'need_review': False} 

365 FACIAL TISSUE {'class_id': 5, 'category': 'Perlengkapan Operasional', 'confidence': 0.95, 'need_review': False} 

PULPEN STANDARD {'class_id': 0, 'category': 'ATK/Administrasi', 'confidence': 0.99, 'need_review': False} 

KERANJANG SAMPAH 50L {'class_id': 5, 'category': 'Perlengkapan Operasional', 'confidence': 1.0, 'need_review': False} 

WALLS PADLE POP 55ML LEMON TEA/36 {'class_id': 4, 'category': 'Minuman & Bahan Minuman', 'confidence': 0.61, 'need_review': True} 

INDOMI SOTO MIE {'class_id': 3, 'category': 'Makanan & Bahan Makanan', 'confidence': 1.0, 'need_review': False} 

INDOMIESOTO MIE {'class_id': 3, 'category': 'Makanan & Bahan Makanan', 'c